In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/diagnostico-de-cancer-de-mama-wisconsin/brca_test.csv
/kaggle/input/competitions/diagnostico-de-cancer-de-mama-wisconsin/csv_example.csv
/kaggle/input/competitions/diagnostico-de-cancer-de-mama-wisconsin/brca_train.csv


# Exploración de datos (EDA)

In [2]:
#Cargo la base de datos
import pandas as pd

# Cargamos el archivo de entrenamiento
df_train = pd.read_csv('/kaggle/input/competitions/diagnostico-de-cancer-de-mama-wisconsin/brca_train.csv')

# Vemos las primeras 5 filas
print(df_train.head())
print(df_train.info())



     id  x.radius_mean  x.texture_mean  x.perimeter_mean  x.area_mean  \
0  2238          14.80           17.66             95.88        674.8   
1  9365          15.78           22.91            105.70        782.6   
2  2932          14.99           25.20             95.54        698.8   
3  3181          19.44           18.82            128.10       1167.0   
4  3159          11.94           20.76             77.87        441.0   

   x.smoothness_mean  x.compactness_mean  x.concavity_mean  \
0            0.09179             0.08890           0.04069   
1            0.11550             0.17520           0.21330   
2            0.09387             0.05131           0.02398   
3            0.10890             0.14480           0.22560   
4            0.08605             0.10110           0.06574   

   x.concave_pts_mean  x.symmetry_mean  ...  x.texture_worst  \
0             0.02260           0.1893  ...            22.74   
1             0.09479           0.2096  ...            30.50

In [3]:
# Revisamos si hay valores faltantes en el set de entrenamiento
print(df_train.isnull().sum())

id                     0
x.radius_mean          0
x.texture_mean         0
x.perimeter_mean       0
x.area_mean            0
x.smoothness_mean      0
x.compactness_mean     0
x.concavity_mean       0
x.concave_pts_mean     0
x.symmetry_mean        0
x.fractal_dim_mean     0
x.radius_se            0
x.texture_se           0
x.perimeter_se         0
x.area_se              0
x.smoothness_se        0
x.compactness_se       0
x.concavity_se         0
x.concave_pts_se       0
x.symmetry_se          0
x.fractal_dim_se       0
x.radius_worst         0
x.texture_worst        0
x.perimeter_worst      0
x.area_worst           0
x.smoothness_worst     0
x.compactness_worst    0
x.concavity_worst      0
x.concave_pts_worst    0
x.symmetry_worst       0
x.fractal_dim_worst    0
Diagnosis              0
dtype: int64


In [4]:
#Identifico target y evalúo desvalance de etiquetas
df_train['Diagnosis'].value_counts()

Diagnosis
0    322
1    191
Name: count, dtype: int64

# Preparación de datos

In [5]:
#Definimos la variable objetivo 
df_train['Diagnosis'].value_counts()# Definimos X (eliminando las columnas que no son características médicas)
X = df_train.drop(columns=['id', 'Diagnosis'])

# Definimos y (nuestra respuesta o target)
y = df_train['Diagnosis']

print("Forma de X:", X.shape)
print("Forma de y:", y.shape)

Forma de X: (513, 30)
Forma de y: (513,)


In [6]:
#divido el df en un nuevo test/train 80/20
from sklearn.model_selection import train_test_split

# Dividimos los datos: 80% para entrenar y 20% para validar
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Datos para entrenar: {len(X_train)}")
print(f"Datos para validar: {len(X_val)}")

Datos para entrenar: 410
Datos para validar: 103


In [7]:
#Escalar los datos
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# "Aprendemos" el promedio y la escala solo de los datos de entrenamiento
X_train_scaled = scaler.fit_transform(X_train)

# Aplicamos esa misma escala a los datos de validación
X_val_scaled = scaler.transform(X_val)

# Entrenar distintos tipo de modelos. 
Voy a seguir el siguiente esquema:

*Modelos Base (Datos Originales)
 		*modeloRL 
 		*modeloRF
 		*modeloSVM
 		*modeloAB
 		*modeloXGB
*Modelos con PCA
		*modeloRLPCA 
 		*modeloRFPCA
 		*modeloSVMPCA
 		*modeloABPCA
 		*modeloXGBPCA
 	

# Modelos Base (Datos Originales)

In [8]:
#REGRESION LOGISTICA

from sklearn.linear_model import LogisticRegression

# Creamos el modelo
modeloRL = LogisticRegression()

# ¡A entrenar! El modelo busca patrones entre X_train y las respuestas y_train
modeloRL.fit(X_train_scaled, y_train)

from sklearn.metrics import classification_report, confusion_matrix

# El modelo hace sus predicciones
predicciones = modeloRL.predict(X_val_scaled)

# Imprimimos los resultados
print("--- Informe de Clasificación RL ---")
print(classification_report(y_val, predicciones))

--- Informe de Clasificación RL ---
              precision    recall  f1-score   support

           0       0.98      0.98      0.98        56
           1       0.98      0.98      0.98        47

    accuracy                           0.98       103
   macro avg       0.98      0.98      0.98       103
weighted avg       0.98      0.98      0.98       103



In [9]:
import pandas as pd
import numpy as np

# 1. Obtenemos los coeficientes del modelo
# Usamos np.abs para ver la magnitud (qué tanto importa) sin importar si es positivo o negativo
importancias_rl = pd.Series(np.abs(modeloRL.coef_[0]), index=X.columns)

print("\n--- Top 5 Variables más importantes según RL ---")
print(importancias_rl.nlargest(5))


--- Top 5 Variables más importantes según RL ---
x.radius_se          1.366640
x.concavity_worst    1.013860
x.texture_worst      0.982268
x.fractal_dim_se     0.949589
x.radius_worst       0.909246
dtype: float64


In [10]:
#Aquí creo una lista de los modelos para poder compararlos unos con otros.
from sklearn.metrics import f1_score

# Creamos un lugar para guardar los resultados
resultados_f1 = {}

# Ejemplo con tu Regresión Logística
pred_rl = modeloRL.predict(X_val_scaled)
resultados_f1['Regresión Logística'] = f1_score(y_val, pred_rl)


In [11]:
#RANDOM FOREST
from sklearn.ensemble import RandomForestClassifier

# 1. Creamos el modelo
# n_estimators=100 significa que usaremos 100 árboles de decisión
modeloRF = RandomForestClassifier(n_estimators=100, random_state=42)

# 2. El bosque analiza las características y aprende a votar si el tumor es M o B
modeloRF.fit(X_train_scaled, y_train)

# 3. El modelo hace sus predicciones sobre los datos de validación
pred_rf = modeloRF.predict(X_val_scaled)

# 4. Imprimimos los resultados para comparar visualmente
print("--- Informe de Clasificación RF ---")
print(classification_report(y_val, pred_rf))

# 5. Guardamos el resultado en tu diccionario comparativo
# Usamos el f1_score que ya importaste antes
resultados_f1['Random Forest'] = f1_score(y_val, pred_rf)

--- Informe de Clasificación RF ---
              precision    recall  f1-score   support

           0       0.95      0.98      0.96        56
           1       0.98      0.94      0.96        47

    accuracy                           0.96       103
   macro avg       0.96      0.96      0.96       103
weighted avg       0.96      0.96      0.96       103



In [12]:
# Ver las 5 variables más importantes
importancias = pd.Series(modeloRF.feature_importances_, index=X.columns)
print("\n--- Top 5 Variables más importantes según RF ---")
print(importancias.nlargest(5))


--- Top 5 Variables más importantes según RF ---
x.area_worst           0.185925
x.concave_pts_worst    0.125113
x.perimeter_worst      0.088273
x.concave_pts_mean     0.085413
x.radius_worst         0.084988
dtype: float64


In [13]:
from sklearn.svm import SVC

# 1. Creamos el modelo
# Usamos el kernel 'rbf' que es el estándar y muy potente para datos complejos
modeloSVM = SVC(kernel='rbf', random_state=42)

# 2. El modelo busca el hiperplano óptimo para separar las clases
modeloSVM.fit(X_train_scaled, y_train)

# 3. El modelo hace sus predicciones
pred_svm = modeloSVM.predict(X_val_scaled)

# 4. Imprimimos los resultados
print("--- Informe de Clasificación SVM ---")
print(classification_report(y_val, pred_svm))

# 5. Guardamos el resultado en tu diccionario comparativo
resultados_f1['SVM'] = f1_score(y_val, pred_svm)

--- Informe de Clasificación SVM ---
              precision    recall  f1-score   support

           0       0.96      0.98      0.97        56
           1       0.98      0.96      0.97        47

    accuracy                           0.97       103
   macro avg       0.97      0.97      0.97       103
weighted avg       0.97      0.97      0.97       103



In [14]:
# Comparación actualizada
for nombre, puntaje in resultados_f1.items():
    print(f"{nombre}: F1-Score = {puntaje:.4f}")

Regresión Logística: F1-Score = 0.9787
Random Forest: F1-Score = 0.9565
SVM: F1-Score = 0.9677


In [15]:
# MODELO ADABOOST
from sklearn.ensemble import AdaBoostClassifier

# 1. Creamos el modelo
# Usamos 100 estimadores para darle suficiente oportunidad de aprender de los errores
modeloAB = AdaBoostClassifier(n_estimators=100, random_state=42)

# 2. El modelo aprenderá secuencialmente de sus propios fallos
modeloAB.fit(X_train_scaled, y_train)

# 3. El modelo hace sus predicciones
pred_ab = modeloAB.predict(X_val_scaled)

# 4. Imprimimos los resultados
print("--- Informe de Clasificación AB ---")
print(classification_report(y_val, pred_ab))

# 5. Guardamos el resultado en tu diccionario comparativo
resultados_f1['AdaBoost'] = f1_score(y_val, pred_ab)

--- Informe de Clasificación AB ---
              precision    recall  f1-score   support

           0       0.96      0.98      0.97        56
           1       0.98      0.96      0.97        47

    accuracy                           0.97       103
   macro avg       0.97      0.97      0.97       103
weighted avg       0.97      0.97      0.97       103



In [16]:
# Ver las 5 variables más importantes para AdaBoost
importancias_ab = pd.Series(modeloAB.feature_importances_, index=X.columns)
print("\n--- Top 5 Variables más importantes según AB ---")
print(importancias_ab.nlargest(5))


--- Top 5 Variables más importantes según AB ---
x.texture_worst        0.102530
x.concave_pts_worst    0.091042
x.radius_se            0.087990
x.area_se              0.058908
x.perimeter_worst      0.054075
dtype: float64


In [17]:
#MODELO XGBBOOST

from xgboost import XGBClassifier

# 1. Creamos el modelo
# Usamos parámetros estándar que suelen funcionar muy bien
modeloXGB = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42)

# 2. XGBoost es muy eficiente y busca minimizar el error de forma extrema
modeloXGB.fit(X_train_scaled, y_train)

# 3. El modelo hace sus predicciones
pred_xgb = modeloXGB.predict(X_val_scaled)

# 4. Imprimimos los resultados
print("--- Informe de Clasificación XGB ---")
print(classification_report(y_val, pred_xgb))

# 5. Guardamos el resultado en tu diccionario comparativo
resultados_f1['XGBoost'] = f1_score(y_val, pred_xgb)

--- Informe de Clasificación XGB ---
              precision    recall  f1-score   support

           0       0.96      0.98      0.97        56
           1       0.98      0.96      0.97        47

    accuracy                           0.97       103
   macro avg       0.97      0.97      0.97       103
weighted avg       0.97      0.97      0.97       103



In [18]:
# Ver las 5 variables más importantes para XGBoost
importancias_xgb = pd.Series(modeloXGB.feature_importances_, index=X.columns)
print("\n--- Top 5 Variables más importantes según XGB ---")
print(importancias_xgb.nlargest(5))


--- Top 5 Variables más importantes según XGB ---
x.perimeter_worst      0.621290
x.radius_worst         0.056474
x.concave_pts_mean     0.054554
x.concave_pts_worst    0.041535
x.area_worst           0.033270
dtype: float32


In [19]:
#Tabla de posiciones final (Sin PCA)
df_comparativo = pd.DataFrame.from_dict(resultados_f1, orient='index', columns=['F1-Score'])
print("--- RANKING FINAL: MODELOS BASE ---")
print(df_comparativo.sort_values(by='F1-Score', ascending=False))

--- RANKING FINAL: MODELOS BASE ---
                     F1-Score
Regresión Logística  0.978723
SVM                  0.967742
AdaBoost             0.967742
XGBoost              0.967742
Random Forest        0.956522


# Modelos con PCA

In [20]:
#Crear la transformación PCA
from sklearn.decomposition import PCA

# 1. Creamos el objeto PCA para que conserve el 95% de la información
pca = PCA(n_components=0.95)

# 2. Ajustamos y transformamos los datos de entrenamiento
X_train_pca = pca.fit_transform(X_train_scaled)

# 3. Transformamos los datos de validación (usando lo que aprendió de train)
X_val_pca = pca.transform(X_val_scaled)

print(f"Número de variables originales: {X_train_scaled.shape[1]}")
print(f"Número de componentes tras PCA: {X_train_pca.shape[1]}")

Número de variables originales: 30
Número de componentes tras PCA: 10


In [21]:
# --- MODELO RL CON PCA ---

# 1. Creamos el modelo
modeloRLPCA = LogisticRegression()

# 2. Entrenamos con los datos transformados por PCA
modeloRLPCA.fit(X_train_pca, y_train)

# 3. Predicciones
pred_rl_pca = modeloRLPCA.predict(X_val_pca)

# 4. Informe
print("--- Informe de Clasificación RL PCA ---")
print(classification_report(y_val, pred_rl_pca))

# 5. Guardamos en el diccionario (usamos un nombre distinto para diferenciarlo)
resultados_f1['Regresión Logística PCA'] = f1_score(y_val, pred_rl_pca)

--- Informe de Clasificación RL PCA ---
              precision    recall  f1-score   support

           0       0.98      0.98      0.98        56
           1       0.98      0.98      0.98        47

    accuracy                           0.98       103
   macro avg       0.98      0.98      0.98       103
weighted avg       0.98      0.98      0.98       103



In [22]:
#MODELO RANDOM FOREST CON PCA
from sklearn.ensemble import RandomForestClassifier

# 1. Creamos el modelo
modeloRFPCA = RandomForestClassifier(n_estimators=100, random_state=42)

# 2. Entrenamos con los datos de la PCA
modeloRFPCA.fit(X_train_pca, y_train)

# 3. Predicciones sobre el set de validación transformado
pred_rf_pca = modeloRFPCA.predict(X_val_pca)

# 4. Informe
print("--- Informe de Clasificación RF PCA ---")
print(classification_report(y_val, pred_rf_pca))

# 5. Guardamos en el diccionario
resultados_f1['Random Forest PCA'] = f1_score(y_val, pred_rf_pca)

--- Informe de Clasificación RF PCA ---
              precision    recall  f1-score   support

           0       0.97      1.00      0.98        56
           1       1.00      0.96      0.98        47

    accuracy                           0.98       103
   macro avg       0.98      0.98      0.98       103
weighted avg       0.98      0.98      0.98       103



In [23]:
#MODELO SVM CON PCA
from sklearn.svm import SVC

# 1. Creamos el modelo
# Seguimos usando el kernel 'rbf' para que la comparación sea justa con el modelo base
modeloSVMPCA = SVC(kernel='rbf', random_state=42)

# 2. Entrenamos con los datos transformados por PCA (los 10 componentes)
modeloSVMPCA.fit(X_train_pca, y_train)

# 3. El modelo hace sus predicciones sobre el set de validación PCA
pred_svm_pca = modeloSVMPCA.predict(X_val_pca)

# 4. Imprimimos los resultados
print("--- Informe de Clasificación SVM PCA ---")
print(classification_report(y_val, pred_svm_pca))

# 5. Guardamos el resultado en el diccionario comparativo
resultados_f1['SVM PCA'] = f1_score(y_val, pred_svm_pca)

--- Informe de Clasificación SVM PCA ---
              precision    recall  f1-score   support

           0       0.97      1.00      0.98        56
           1       1.00      0.96      0.98        47

    accuracy                           0.98       103
   macro avg       0.98      0.98      0.98       103
weighted avg       0.98      0.98      0.98       103



In [24]:
#MODELO ADABOOST CON PCA
from sklearn.ensemble import AdaBoostClassifier

# 1. Creamos el modelo
# Mantenemos los 100 estimadores para que la comparación sea justa con la versión sin PCA
modeloABPCA = AdaBoostClassifier(n_estimators=100, random_state=42)

# 2. Entrenamos con los datos transformados por PCA
modeloABPCA.fit(X_train_pca, y_train)

# 3. El modelo hace sus predicciones sobre el set de validación PCA
pred_ab_pca = modeloABPCA.predict(X_val_pca)

# 4. Imprimimos los resultados
print("--- Informe de Clasificación AB PCA ---")
print(classification_report(y_val, pred_ab_pca))

# 5. Guardamos el resultado en el diccionario comparativo
resultados_f1['AdaBoost PCA'] = f1_score(y_val, pred_ab_pca)

--- Informe de Clasificación AB PCA ---
              precision    recall  f1-score   support

           0       0.95      0.98      0.96        56
           1       0.98      0.94      0.96        47

    accuracy                           0.96       103
   macro avg       0.96      0.96      0.96       103
weighted avg       0.96      0.96      0.96       103



In [25]:
#MODELO XGBOOST CON PCA
from xgboost import XGBClassifier

# 1. Creamos el modelo
# Usamos los mismos parámetros que en la versión base para que sea una comparativa justa
modeloXGBPCA = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42)

# 2. Entrenamos con los datos transformados por PCA
modeloXGBPCA.fit(X_train_pca, y_train)

# 3. El modelo hace sus predicciones sobre el set de validación PCA
pred_xgb_pca = modeloXGBPCA.predict(X_val_pca)

# 4. Imprimimos los resultados
print("--- Informe de Clasificación XGB PCA ---")
print(classification_report(y_val, pred_xgb_pca))

# 5. Guardamos el resultado en el diccionario comparativo
resultados_f1['XGBoost PCA'] = f1_score(y_val, pred_xgb_pca)

--- Informe de Clasificación XGB PCA ---
              precision    recall  f1-score   support

           0       0.98      0.96      0.97        56
           1       0.96      0.98      0.97        47

    accuracy                           0.97       103
   macro avg       0.97      0.97      0.97       103
weighted avg       0.97      0.97      0.97       103



# RANKING FINAL DE TODOS LOS MODELOS

In [26]:
# Convertimos el diccionario en un DataFrame para que se vea como una tabla de Excel
df_resultados = pd.DataFrame.from_dict(resultados_f1, orient='index', columns=['F1-Score'])

# Ordenamos de mayor a menor puntaje
df_resultados = df_resultados.sort_values(by='F1-Score', ascending=False)

print("--- RANKING FINAL DE TODOS LOS MODELOS ---")
print(df_resultados)

--- RANKING FINAL DE TODOS LOS MODELOS ---
                         F1-Score
Regresión Logística      0.978723
Regresión Logística PCA  0.978723
SVM PCA                  0.978261
Random Forest PCA        0.978261
XGBoost PCA              0.968421
XGBoost                  0.967742
SVM                      0.967742
AdaBoost                 0.967742
Random Forest            0.956522
AdaBoost PCA             0.956522


# Código para crear el archivo para kaggle según el modelo elegido

In [27]:
#Código para crear el archivo para kaggle para modelos base.

# 1. Cargamos el archivo de test oficial de Kaggle
df_test_final = pd.read_csv('/kaggle/input/competitions/diagnostico-de-cancer-de-mama-wisconsin/brca_test.csv')

# 2. Guardamos los IDs para el archivo de salida
ids_test = df_test_final['id']

# 3. Preparamos las características (quitamos el ID)
X_test_oficial = df_test_final.drop(columns=['id'])

# 4. Escalamos los datos usando el mismo scaler de antes (¡Muy importante!)
X_test_oficial_scaled = scaler.transform(X_test_oficial)

# 5. Predecimos con modelo elegido (Regresión Logística)
predicciones_RL = modeloRL.predict(X_test_oficial_scaled)

# 6. Creamos el DataFrame de entrega
submission_RL = pd.DataFrame({
    'id': ids_test,
    'Diagnosis': predicciones_RL
})

# 7. Guardamos el archivo .csv
submission_RL.to_csv('submission.csv', index=False)

# Verificamos rápidamente las primeras filas
print("--- Vista previa del archivo RL ---")
print(submission_RL.head())
print("¡Archivo listo para descargar y subir a Kaggle!")

--- Vista previa del archivo RL ---
     id  Diagnosis
0  3721          1
1  4893          0
2  3602          1
3  3271          1
4  4091          0
¡Archivo listo para descargar y subir a Kaggle!


In [28]:
# 1.#Código para crear el archivo para kaggle para modelos PCA

# Cargamos el archivo de test oficial
# Asegúrate de que la ruta sea la correcta para tu competencia
df_test_final = pd.read_csv('/kaggle/input/competitions/diagnostico-de-cancer-de-mama-wisconsin/brca_test.csv')

# 2. Guardamos los IDs originales
ids_test = df_test_final['id']

# 3. Preparamos las características (quitamos la columna ID)
X_test_oficial = df_test_final.drop(columns=['id'])

# 4. Escalamiento: Usamos el MISMO 'scaler' que entrenamos con los datos de train
X_test_scaled = scaler.transform(X_test_oficial)

# 5. Transformación PCA: Usamos el MISMO objeto 'pca' para reducir a 10 componentes
X_test_pca = pca.transform(X_test_scaled)

# 6. Predicción: Usamos tu modelo 'modeloSVMPCA'
predicciones_svm_pca = modeloSVMPCA.predict(X_test_pca)

# 7. Creamos el DataFrame con el formato de la competencia (id, Diagnosis)
submission_svm = pd.DataFrame({
    'id': ids_test,
    'Diagnosis': predicciones_svm_pca
})

# 8. Guardamos el archivo .csv 
submission_svm.to_csv('submission.csv', index=False)

# Verificamos rápidamente las primeras filas
print("--- Vista previa del archivo SVM PCA ---")
print(submission_svm.head())

--- Vista previa del archivo SVM PCA ---
     id  Diagnosis
0  3721          1
1  4893          0
2  3602          1
3  3271          1
4  4091          0


In [29]:
# 1.#Código para crear el archivo para kaggle para modelos PCA

# Cargamos el archivo de test oficial
# Asegúrate de que la ruta sea la correcta para tu competencia
df_test_final = pd.read_csv('/kaggle/input/competitions/diagnostico-de-cancer-de-mama-wisconsin/brca_test.csv')

# 2. Guardamos los IDs originales
ids_test = df_test_final['id']

# 3. Preparamos las características (quitamos la columna ID)
X_test_oficial = df_test_final.drop(columns=['id'])

# 4. Escalamiento: Usamos el MISMO 'scaler' que entrenamos con los datos de train
X_test_scaled = scaler.transform(X_test_oficial)

# 5. Transformación PCA: Usamos el MISMO objeto 'pca' para reducir a 10 componentes
X_test_pca = pca.transform(X_test_scaled)

# 6. Predicción: Usamos tu modelo 'modeloXGBPCA'
predicciones_xgb_pca = modeloXGBPCA.predict(X_test_pca)

# 7. Creamos el DataFrame con el formato de la competencia (id, Diagnosis)
submission_xgb_pca = pd.DataFrame({
    'id': ids_test,
    'Diagnosis': predicciones_xgb_pca
})

# 8. Guardamos el archivo .csv 
submission_xgb_pca.to_csv('submission.csv', index=False)

# Verificamos rápidamente las primeras filas
print("--- Vista previa del archivo XGB PCA ---")
print(submission_xgb_pca.head())

--- Vista previa del archivo XGB PCA ---
     id  Diagnosis
0  3721          1
1  4893          0
2  3602          1
3  3271          1
4  4091          0


In [30]:
#Acá se me ocurrió realizar un código que haga una iteración sobre los modelos que hice para sacar todos los .csv de una vez y reducir el riesgo de error de tipeo. Para esto se crea un diccionario de modelos

# 1. Definimos un diccionario con la configuración de cada modelo
# El formato es: 'Nombre_Archivo': (objeto_modelo, usa_pca)
modelos_a_procesar = {
    'RL_Base': (modeloRL, False),
    'RL_PCA': (modeloRLPCA, True),
    'SVM_Base': (modeloSVM, False),
    'SVM_PCA': (modeloSVMPCA, True),
    'RF_Base': (modeloRF, False),
    'RF_PCA': (modeloRFPCA, True),
    'Ada_Base': (modeloAB, False),
    'Ada_PCA': (modeloABPCA, True),
    'XGB_Base': (modeloXGB, False),
    'XGB_PCA': (modeloXGBPCA, True),
    }

print(f"Iniciando generación automática de {len(modelos_a_procesar)} archivos...")

# 2. El bucle mágico
for nombre, (modelo, usa_pca) in modelos_a_procesar.items():
    
    # Seleccionamos los datos correctos según el modelo
    datos_entrada = X_test_pca if usa_pca else X_test_oficial_scaled
    
    # Realizamos la predicción
    predicciones = modelo.predict(datos_entrada)
    
    # Creamos el DataFrame de envío
    df_sub = pd.DataFrame({
        'id': ids_test,
        'Diagnosis': predicciones
    })
    
    # Guardamos con un nombre único
    nombre_archivo = f'submission_{nombre}.csv'
    df_sub.to_csv(nombre_archivo, index=False)
    
    print(f"Archivo generado: {nombre_archivo}")

print("\n¡Proceso completado! Revisa la carpeta 'Output' en el panel derecho.")

Iniciando generación automática de 10 archivos...
Archivo generado: submission_RL_Base.csv
Archivo generado: submission_RL_PCA.csv
Archivo generado: submission_SVM_Base.csv
Archivo generado: submission_SVM_PCA.csv
Archivo generado: submission_RF_Base.csv
Archivo generado: submission_RF_PCA.csv
Archivo generado: submission_Ada_Base.csv
Archivo generado: submission_Ada_PCA.csv
Archivo generado: submission_XGB_Base.csv
Archivo generado: submission_XGB_PCA.csv

¡Proceso completado! Revisa la carpeta 'Output' en el panel derecho.


In [31]:
#entrenar modelo GMM

from sklearn.mixture import GaussianMixture
from sklearn.metrics import classification_report, f1_score

# Separamos los datos de entrenamiento según la clase (usando tus datos escalados)
X_train_0 = X_train_scaled[y_train == 0]
X_train_1 = X_train_scaled[y_train == 1]

# Creamos el modelo GMM para cada clase
# Usamos n_components=1 para representar cada grupo como una campana de Gauss única
modeloGMM_0 = GaussianMixture(n_components=1, random_state=42).fit(X_train_0)
modeloGMM_1 = GaussianMixture(n_components=1, random_state=42).fit(X_train_1)

# Función de decisión basada en máxima verosimilitud (Likelihood)
def predict_gmm(X):
    # Comparamos qué tan probable es que el dato pertenezca a la nube 0 o a la 1
    p0 = modeloGMM_0.score_samples(X)
    p1 = modeloGMM_1.score_samples(X)
    return (p1 > p0).astype(int)

print("✅ Modelos GMM entrenados por clase.")

✅ Modelos GMM entrenados por clase.


In [32]:
# Realizamos la predicción sobre tu set de validación
pred_gmm_val = predict_gmm(X_val_scaled)

# Generamos el informe
print("--- Informe de Clasificación: GMM (Validación Local) ---")
print(classification_report(y_val, pred_gmm_val))

# Guardamos el resultado en tu diccionario de F1 si lo estás usando
# f1_gmm = f1_score(y_val, pred_gmm_val)

--- Informe de Clasificación: GMM (Validación Local) ---
              precision    recall  f1-score   support

           0       0.95      0.96      0.96        56
           1       0.96      0.94      0.95        47

    accuracy                           0.95       103
   macro avg       0.95      0.95      0.95       103
weighted avg       0.95      0.95      0.95       103



In [33]:
# 1. Predicción sobre el set de test oficial (escalado)
pred_gmm_test = predict_gmm(X_test_oficial_scaled)

# 2. Creamos el DataFrame siguiendo tu formato
submission_GMM = pd.DataFrame({
    'id': ids_test,
    'Diagnosis': pred_gmm_test
})

# 3. Guardamos el archivo .csv
nombre_archivo_gmm = 'submission_GMM.csv'
submission_GMM.to_csv(nombre_archivo_gmm, index=False)

print(f"✅ Archivo generado: {nombre_archivo_gmm}")
print(submission_GMM.head())

✅ Archivo generado: submission_GMM.csv
     id  Diagnosis
0  3721          1
1  4893          0
2  3602          1
3  3271          1
4  4091          0
